# Pokepaste Scraper
Scrapes team data from pokepaste links in the VGCPastes Repository Excel file.

In [4]:
import openpyxl
import requests
from bs4 import BeautifulSoup
import pickle
import time
import re
import os

In [5]:
# ========================== TOGGLES ==========================
# Set to True to only scrape rows where the EVs column is "Yes"
# Set to False to scrape ALL rows with a pokepaste link
ONLY_WITH_EVS = False

# Set to True to include EV entries in each pokemon vector
# Set to False to omit the 6 EV fields from the output
INCLUDE_EVS = False
# =============================================================

In [6]:
# Resolve paths relative to this notebook's directory
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
EXCEL_PATH = os.path.join(NOTEBOOK_DIR, "VGCPastes Repository.xlsx")
SHEET_NAME = "Champions M-A"
OUTPUT_PATH = os.path.join(NOTEBOOK_DIR, "team_vectors.pkl")

wb = openpyxl.load_workbook(EXCEL_PATH, read_only=True, data_only=True)
ws = wb[SHEET_NAME]

paste_urls = []
for row in ws.iter_rows(min_row=4, values_only=True):
    url = row[24]   # Column Y (pokepaste link)
    evs = row[25]   # Column Z (EVs flag)
    if not url or not str(url).startswith('http'):
        continue
    if ONLY_WITH_EVS and (not evs or str(evs).strip().lower() != 'yes'):
        continue
    paste_urls.append(str(url).strip())

wb.close()
print(f"Found {len(paste_urls)} pokepaste URLs to scrape")

Found 769 pokepaste URLs to scrape


In [7]:
EV_KEYS = ['HP', 'Atk', 'Def', 'SpA', 'SpD', 'Spe']

def parse_pokemon_block(text):
    """Parse a single pokemon's text block into the desired vector."""
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return None

    # Line 1: Name @ Item
    first_line = lines[0]
    if ' @ ' in first_line:
        name, item = first_line.split(' @ ', 1)
    else:
        name = first_line
        item = 'NONE'

    ability = 'NONE'
    evs = {k: 0 for k in EV_KEYS}
    moves = []

    for line in lines[1:]:
        if line.startswith('Ability:'):
            ability = line.split('Ability:', 1)[1].strip()
        elif line.startswith('EVs:'):
            ev_str = line.split('EVs:', 1)[1].strip()
            for part in ev_str.split('/'):
                part = part.strip()
                match = re.match(r'(\d+)\s+(\w+)', part)
                if match:
                    val, stat = int(match.group(1)), match.group(2)
                    if stat in evs:
                        evs[stat] = val
        elif line.startswith('- '):
            moves.append(line[2:].strip())

    # Pad moves to 4 with NONE
    while len(moves) < 4:
        moves.append('NONE')
    moves = moves[:4]

    if INCLUDE_EVS:
        return [
            name.strip(), ability,
            evs['HP'], evs['Atk'], evs['Def'],
            evs['SpA'], evs['SpD'], evs['Spe'],
            item.strip(),
            moves[0], moves[1], moves[2], moves[3]
        ]
    else:
        return [
            name.strip(), ability,
            item.strip(),
            moves[0], moves[1], moves[2], moves[3]
        ]

In [8]:
def scrape_pokepaste(url):
    """Scrape a pokepaste URL and return a list of 6 pokemon vectors."""
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    pokemon_blocks = []
    for pre in soup.find_all('pre'):
        text = pre.get_text()
        if text.strip():
            pokemon_blocks.append(text)

    team = []
    for block in pokemon_blocks:
        parsed = parse_pokemon_block(block)
        if parsed:
            team.append(parsed)

    return team

In [9]:
all_teams = []
failed_urls = []

for i, url in enumerate(paste_urls):
    if (i + 1) % 50 == 0 or i == 0:
        print(f"Scraping {i + 1}/{len(paste_urls)}: {url}")
    try:
        team = scrape_pokepaste(url)
        if team:
            all_teams.append(team)
        else:
            failed_urls.append((url, 'empty parse'))
    except Exception as e:
        failed_urls.append((url, str(e)))
    time.sleep(0.3)  # be polite to the server

print(f"\nDone! Scraped {len(all_teams)} teams successfully.")
if failed_urls:
    print(f"Failed on {len(failed_urls)} URLs:")
    for url, reason in failed_urls[:10]:
        print(f"  {url} — {reason}")

Scraping 1/769: https://pokepast.es/ae516ffce0fe7251
Scraping 50/769: https://pokepast.es/b4e718f6136340de
Scraping 100/769: https://pokepast.es/121d5a71ecaea08d
Scraping 150/769: https://pokepast.es/f11a71524699880d
Scraping 200/769: https://pokepast.es/bcdad0fa3c1544eb
Scraping 250/769: https://pokepast.es/ca4084618a5f2e82
Scraping 300/769: https://pokepast.es/3cd2f97e3040769d
Scraping 350/769: https://pokepast.es/a1ea30499afd4fa2
Scraping 400/769: https://pokepast.es/22d85b73d74b58d2
Scraping 450/769: https://pokepast.es/d58aea0999b81958
Scraping 500/769: https://pokepast.es/9c728f17de3f561d
Scraping 550/769: https://pokepast.es/a1c66ba27531f3cf
Scraping 600/769: https://pokepast.es/b7b6810f8d31fdba
Scraping 650/769: https://pokepast.es/4ddae7ca44292e97
Scraping 700/769: https://pokepast.es/861e1cb473a4f94f
Scraping 750/769: https://pokepast.es/f18c096e9fe43183

Done! Scraped 769 teams successfully.


In [10]:
# Preview first team
if all_teams:
    print(f"Example team (first scraped):")
    for mon in all_teams[0]:
        print(f"  {mon}")

Example team (first scraped):
  ['Aerodactyl', 'Unnerve', 'Aerodactylite', 'Rock Slide', 'Dual Wingbeat', 'Tailwind', 'Protect']
  ['Basculegion', 'Adaptability', 'Kasib Berry', 'Last Respects', 'Wave Crash', 'Aqua Jet', 'Protect']
  ['Kingambit', 'Defiant', 'Chople Berry', 'Kowtow Cleave', 'Low Kick', 'Sucker Punch', 'Protect']
  ['Floette-Eternal', 'Flower Veil', 'Floettite', 'Moonblast', 'Dazzling Gleam', 'Light of Ruin', 'Protect']
  ['Sneasler', 'Poison Touch', 'Focus Sash', 'Close Combat', 'Dire Claw', 'Fake Out', 'Protect']
  ['Garchomp', 'Rough Skin', 'Sitrus Berry', 'Dragon Claw', 'Earthquake', 'Rock Slide', 'Protect']


In [11]:
with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(all_teams, f)

print(f"Saved {len(all_teams)} teams to {OUTPUT_PATH}")
print(f"Structure: list of {len(all_teams)} teams, each team is a list of pokemon vectors")
if all_teams:
    print(f"Each pokemon vector has {len(all_teams[0][0])} entries")

Saved 769 teams to /Users/xanderdeanhardt/Documents/Claude Code Folder/Data_science_files/MS ADS/ML 2/Final Project/team_vectors.pkl
Structure: list of 769 teams, each team is a list of pokemon vectors
Each pokemon vector has 7 entries


In [12]:
# Verify reload
with open(OUTPUT_PATH, 'rb') as f:
    loaded = pickle.load(f)
print(f"Reloaded {len(loaded)} teams from {OUTPUT_PATH}")
print(f"First team, first pokemon: {loaded[0][0]}")

Reloaded 769 teams from /Users/xanderdeanhardt/Documents/Claude Code Folder/Data_science_files/MS ADS/ML 2/Final Project/team_vectors.pkl
First team, first pokemon: ['Aerodactyl', 'Unnerve', 'Aerodactylite', 'Rock Slide', 'Dual Wingbeat', 'Tailwind', 'Protect']
